# MKS Germany

Per private communication, the projection is:

````
; Stereographische Projektion vereinbaren, falls gewuenscht
xw=6.2 & xo=15.0 & ys=47.2 & yn=55.1   ; unbedingt so lassen!
IF stereo THEN BEGIN
   MAP_SET, 50., 10.5, $
           LIMIT=[ys,xw, yn, xo], /STEREO, POSITION=[0,0,1,0.85], $
           NOBORDER=1, ISOTROPIC=1, XMARGIN=0, YMARGIN=0
ENDIF ELSE BEGIN
   PLOT, [xw,xo], [ys,yn], /NODATA, XSTYLE=1+4, YSTYLE=1+4+16, $
        XMARGIN=[0.,0.], YMARGIN=[0.,0.]
ENDELSE
````

**Reference**

https://gitlab.dwd.de/ku/libraries/pyku/-/issues/128

## Note

However the projection seems to be extended in pixels by a certain margin. I did not try to calculate it backwards and that would be complicated.


### Calculate

In [ ]:
import pyproj

# Define the geographic coordinates of the lower lef (ll) and upper right (ur) corners
# ------------------------------------------------------------------------------------
lat_ll = 47.2
lon_ll =  6.2
lat_ur = 55.1
lon_ur = 15.0

# Target projection
# -----------------

proj_string = '+proj=stere +lat_0=50 +lat_ts=50 +lon_0=10.5 +x_0=0 +y_0=0 +units=m'

# The source projection are the geographic coordinates on a WGS84 earth
# ---------------------------------------------------------------------

wgs84_latlong_proj = pyproj.Proj(proj='latlong', datum='WGS84')

# Create transformer and transform
# ------------------

transformer = pyproj.Transformer.from_proj(proj_from=wgs84_latlong_proj, proj_to=proj_string)

x_ll, y_ll = transformer.transform(lon_ll, lat_ll)
x_ur, y_ur = transformer.transform(lon_ur, lat_ur)

x_ll, y_ll = round(x_ll, 4), round(y_ll, 4)
x_ur, y_ur = round(x_ur, 4), round(y_ur, 4)

resolution=5000.0
x_size=(x_ur-x_ll)/resolution
y_size=(y_ur-y_ll)/resolution

# Print for comparison
# --------------------

print("\nCalculated\n")
print(proj_string)
print("""\
+------------+-----------+------------+--------------+-------------+
| Coordinate |   lon     |     lat    |     x        |     y       |
+============+===========+============+==============+=============+\
"""
)
print(f"| LowerLeft  | {lon_ll:7.4f}   |  {lat_ll:7.4f}   | {x_ll:-12.4f} | {y_ll:-12.4f}|")
print(f"| UpperRight | {lon_ur:7.4f}   |  {lat_ur:7.4f}   | {x_ur:-12.4f} | {y_ur:-12.4f}|")

lower_left_x, lower_left_y, upper_right_x, upper_right_y = x_ll, y_ll, x_ur, y_ur

### Create area definition for pyku

In [ ]:
from pyresample.geometry import AreaDefinition
area_id = 'mks_germany'
description = 'MKS Germany Stereo Projection'
proj_id = 'mks_germany'
projection = proj_string
area_extent =  (lower_left_x, lower_left_y, upper_right_x, upper_right_y)
area_def = AreaDefinition(
    area_id,
    description,
    proj_id,
    projection,
    x_size,
    y_size,
    area_extent
)

### Area definition for pyku configuration file

In [ ]:
print(area_def.dump())

### Show the map

In [ ]:
area_def.to_cartopy_crs()

In [ ]:
import pyku.resources as resources
ds = resources.get_test_data('global_data').isel(time=0).pyku.project(area_def)
ds.ana.one_map(var='tas', crs=area_def, size_inches=(5, 8))